In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./indobertweet-finetuned"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()

df = pd.read_csv("bersatu predict.csv")

texts = df["clean_textdisplay"].fillna("").astype(str).tolist()

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

batch_size = 32
results = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]

    inputs = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1)

    results.extend(preds.cpu().tolist())

df["sentiment"] = [id2label[p] for p in results]

df.to_csv("hasil_sentimen.predictions.csv", index=False)

print(df[["clean_textdisplay", "sentiment"]].head())

                                   clean_textdisplay sentiment
0  doain bisa kebeli mobil listrik karena udah pu...  positive
1  blue bird aja berani beli electric vehicle dip...  positive
2  jangan lupa catl udah jual baterai garambrbrwa...  positive
3  yg hidup dan tinggal di kampung grand max thr ...  positive
4  electric vehicle bukan segmen seperti kaum men...  positive


In [4]:
test = [
    "ini bagus banget",
    "biasa saja sih",
    "parah banget jelek"
]